# Movie Recommendation System

A content-based movie recommendation system built using the TMDB dataset.

**Upgrades applied:**
- TF-IDF vectorization with NLTK stemming
- Robust `ast.literal_eval` with safe fallback
- Fuzzy title search via `rapidfuzz`
- Hybrid scoring with tunable weights
- NDCG@K + Precision@K + catalog coverage evaluation
- Sparse similarity storage (instead of dense pickle)

---

## Pipeline
1. Load & merge `movies.csv`, `credits.csv`, `poster.csv`
2. Safe feature extraction (genres, keywords, top-3 cast, director)
3. NLTK stemming on combined tags
4. TF-IDF → Cosine Similarity
5. Hybrid scoring with tunable weights
6. Fuzzy title matching
7. NDCG@K + Precision@K evaluation
8. Sparse similarity storage

## 1. Importing Libraries

In [37]:
import pandas as pd
import numpy as np
import ast
import pickle
import warnings
warnings.filterwarnings('ignore')

import nltk
from nltk.stem import PorterStemmer
nltk.download('punkt', quiet=True)

from rapidfuzz import process as rfuzz
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import lil_matrix
import scipy.sparse as sp

ps = PorterStemmer()
print("Libraries loaded.")

Libraries loaded.


## 2. Loading Data

In [38]:
movies  = pd.read_csv('Data/movies.csv')
credits = pd.read_csv('Data/credits.csv')
posters = pd.read_csv('Data/poster.csv')

print(f"Movies  : {movies.shape}")
print(f"Credits : {credits.shape}")
print(f"Posters : {posters.shape}")

Movies  : (4803, 20)
Credits : (4803, 4)
Posters : (9837, 2)


## 3. Merging and Cleaning

In [39]:
movies = movies.merge(credits, on='title')
movies = movies.merge(posters, on='title')

movies = movies[['movie_id', 'title', 'overview',
                 'genres', 'keywords', 'cast', 'crew',
                 'vote_average', 'popularity', 'poster']]

movies.dropna(inplace=True)
movies.reset_index(drop=True, inplace=True)

print(f"Total usable movies: {len(movies)}")
movies.head(2)

Total usable movies: 3028


,movie_id,title,overview,genres,keywords,cast,crew,vote_average,popularity,poster
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de...",7.2,150.437577,https://image.tmdb.org/t/p/original/jRXYjXNq0C...
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de...",6.9,139.082615,https://image.tmdb.org/t/p/original/2YMnBRh8F6...


## 4. Safe Feature Extraction

All `ast.literal_eval` calls are wrapped in a `safe_parse()` fallback so malformed
JSON rows silently return `[]` instead of crashing the pipeline.

In [40]:
def safe_parse(obj, default=None):
    """Safely parse a JSON-like string; return default on failure."""
    if default is None:
        default = []
    try:
        return ast.literal_eval(obj) if isinstance(obj, str) else obj
    except (ValueError, SyntaxError):
        return default

def extract_names(obj):
    return [i['name'].replace(" ", "") for i in safe_parse(obj) if 'name' in i]

def top3_cast(obj):
    return [i['name'].replace(" ", "") for i in safe_parse(obj)[:3] if 'name' in i]

def fetch_director(obj):
    for i in safe_parse(obj):
        if i.get('job') == 'Director':
            return [i['name'].replace(" ", "")]
    return []

movies['genres']   = movies['genres'].apply(extract_names)
movies['keywords'] = movies['keywords'].apply(extract_names)
movies['cast']     = movies['cast'].apply(top3_cast)
movies['crew']     = movies['crew'].apply(fetch_director)
movies['overview'] = movies['overview'].apply(lambda x: x.lower().split())

print("Safe feature extraction complete.")

Safe feature extraction complete.


## 5. Stemming + Building Tags

NLTK `PorterStemmer` collapses word variants (`running`, `runs`, `runner` → `run`),
so the TF-IDF vocabulary is smaller and similarity scores are more accurate.

In [41]:
movies['tags'] = (movies['overview'] + movies['genres'] +
                  movies['keywords'] + movies['cast'] + movies['crew'])

new_df = movies[['movie_id', 'title', 'tags',
                 'vote_average', 'popularity', 'poster',
                 'genres']].copy()               # keep genres for evaluation
new_df.rename(columns={'genres': 'genres_list'}, inplace=True)

def stem_tags(tokens):
    return " ".join(ps.stem(w) for w in tokens)

new_df['tags'] = new_df['tags'].apply(stem_tags)

print("Stemming complete.")
new_df[['title', 'tags']].head(3)

Stemming complete.


,title,tags
0,Avatar,"in the 22nd century, a parapleg marin is dispa..."
1,Pirates of the Caribbean: At World's End,"captain barbossa, long believ to be dead, ha c..."
2,Spectre,a cryptic messag from bond’ past send him on a...


## 6. TF-IDF Vectorization + Cosine Similarity

In [42]:
tfidf = TfidfVectorizer(max_features=10000, stop_words='english')
tfidf_matrix = tfidf.fit_transform(new_df['tags'])
print(f"TF-IDF matrix : {tfidf_matrix.shape}")

similarity = cosine_similarity(tfidf_matrix)
print(f"Similarity mat: {similarity.shape}")

TF-IDF matrix : (3028, 10000)
Similarity mat: (3028, 3028)


## 7. Normalize Signals for Hybrid Scoring

In [43]:
def minmax(series):
    return (series - series.min()) / (series.max() - series.min())

new_df['vote_norm'] = minmax(new_df['vote_average'])
new_df['pop_norm']  = minmax(new_df['popularity'])

vote_arr = new_df['vote_norm'].values
pop_arr  = new_df['pop_norm'].values

print("Normalization complete.")

Normalization complete.


## 8. Fuzzy Title Search

`rapidfuzz.process.extractOne` returns the closest match with a confidence score.
Queries like `'avtar'` and `'dark night'` will be resolved without crashing.

In [44]:
ALL_TITLES = new_df['title'].tolist()

def find_title(query, cutoff=60):
    """Return the best matching title or None if confidence < cutoff."""
    result = rfuzz.extractOne(query, ALL_TITLES, score_cutoff=cutoff)
    if result is None:
        return None
    return result[0]          # (match_string, score, index)

# Quick sanity check
for q in ['avtar', 'dark night', 'interstella', 'Toy Stori']:
    print(f"  '{q}'  →  '{find_title(q)}'")

  'avtar'  →  'Avatar'
  'dark night'  →  'The Dark Knight Rises'
  'interstella'  →  'Interstellar'
  'Toy Stori'  →  'Toy Story'


## 9. Recommendation Function (Tunable Weights)

- **`w_sim`** : weight for cosine similarity (content match)
- **`w_vote`** : weight for normalised vote average
- **`w_pop`**  : weight for normalised popularity

Defaults give a Netflix-like balance: quality content ranked above obscure matches.

In [45]:
def recommend(movie, top_n=5, w_sim=0.70, w_vote=0.20, w_pop=0.10):
    """
    Recommend top-N movies.
    
    Supports fuzzy title matching; weights must sum to 1.
    """
    matched = find_title(movie)
    if matched is None:
        print(f"No close match found for '{movie}'")
        return
    if matched != movie:
        print(f"(Matched '{movie}' → '{matched}')")

    idx    = new_df[new_df['title'] == matched].index[0]
    sim    = similarity[idx]
    scores = sim * w_sim + vote_arr * w_vote + pop_arr * w_pop

    ranked = sorted(
        [(i, s) for i, s in enumerate(scores) if i != idx],
        key=lambda x: x[1], reverse=True
    )[:top_n]

    print(f"\nRecommendations for '{matched}' "
          f"(sim={w_sim}, vote={w_vote}, pop={w_pop}):")
    print("-" * 60)
    for rank, (i, score) in enumerate(ranked, 1):
        row = new_df.iloc[i]
        print(f"{rank}. {row.title:<42} score={score:.4f}")
        print(f"   Poster: {row.poster}")

In [46]:
# Test with typos / partial names
recommend('avtar')                         # fuzzy match → Avatar
print()
recommend('The Dark Knight')
print()
recommend('Interstellar', w_sim=0.5, w_vote=0.3, w_pop=0.2)   # custom weights

(Matched 'avtar' → 'Avatar')

Recommendations for 'Avatar' (sim=0.7, vote=0.2, pop=0.1):
------------------------------------------------------------
1. Aliens                                     score=0.3305
   Poster: https://image.tmdb.org/t/p/original/r1x5JGpyqZU8PYhbs4UcrO1Xb6x.jpg
2. Interstellar                               score=0.3117
   Poster: https://image.tmdb.org/t/p/original/gEU2QniE6E77NI6lCU6MxlNBvIx.jpg
3. Guardians of the Galaxy                    score=0.2957
   Poster: https://image.tmdb.org/t/p/original/r7vmZjiyZw9rpJMQJdXpjgiCOk9.jpg
4. Star Trek Into Darkness                    score=0.2745
   Poster: https://image.tmdb.org/t/p/original/7XrRkhMa9lQ71RszzSyVrJVvhyS.jpg
5. Minions                                    score=0.2665
   Poster: https://image.tmdb.org/t/p/original/vlOgaxUiMOA8sPDG9n3VhQabnEi.jpg


Recommendations for 'The Dark Knight' (sim=0.7, vote=0.2, pop=0.1):
------------------------------------------------------------
1. The Dark Knight Rises     

## 10. Evaluation — Precision@K, NDCG@K & Catalog Coverage

| Metric | What it measures |
|---|---|
| **Precision@K** | Fraction of top-K recs sharing a genre with the query |
| **NDCG@K** | Same, but penalises relevant items ranked lower |
| **Coverage** | % of unique movies surfaced across all test queries |

In [47]:
def get_ranked(movie, k, w_sim=0.70, w_vote=0.20, w_pop=0.10):
    matched = find_title(movie)
    if matched is None:
        return None, None
    idx    = new_df[new_df['title'] == matched].index[0]
    scores = similarity[idx] * w_sim + vote_arr * w_vote + pop_arr * w_pop
    ranked = sorted(
        [(i, s) for i, s in enumerate(scores) if i != idx],
        key=lambda x: x[1], reverse=True
    )[:k]
    query_genres = set(new_df.iloc[idx]['genres_list'])
    return ranked, query_genres

def precision_at_k(movie, k=5, **kwargs):
    ranked, qg = get_ranked(movie, k, **kwargs)
    if ranked is None: return None
    hits = sum(1 for i, _ in ranked if set(new_df.iloc[i]['genres_list']) & qg)
    return hits / k

def ndcg_at_k(movie, k=5, **kwargs):
    ranked, qg = get_ranked(movie, k, **kwargs)
    if ranked is None: return None
    dcg  = sum(
        int(bool(set(new_df.iloc[i]['genres_list']) & qg)) / np.log2(r + 2)
        for r, (i, _) in enumerate(ranked)
    )
    idcg = sum(1 / np.log2(r + 2) for r in range(min(k, len(qg))))
    return dcg / idcg if idcg > 0 else 0.0

def catalog_coverage(movies_list, k=5):
    all_recs = set()
    for m in movies_list:
        ranked, _ = get_ranked(m, k)
        if ranked:
            all_recs.update(i for i, _ in ranked)
    return len(all_recs) / len(new_df)

# Evaluate
test_movies = ['Avatar', 'The Dark Knight', 'Interstellar', 'Toy Story', 'Inception']

print(f"{'Movie':<32} {'P@5':>6} {'NDCG@5':>8}")
print("-" * 50)
for m in test_movies:
    p = precision_at_k(m, k=5)
    n = ndcg_at_k(m, k=5)
    if p is not None:
        print(f"{m:<32} {p:>6.2f} {n:>8.4f}")

cov = catalog_coverage(test_movies, k=5)
print(f"\nCatalog coverage (top-5, {len(test_movies)} queries): {cov*100:.2f}%")

Movie                               P@5   NDCG@5
--------------------------------------------------
Avatar                             1.00   1.1510
The Dark Knight                    1.00   1.1510
Interstellar                       1.00   1.3836
Toy Story                          0.80   1.1815
Inception                          0.80   0.8688

Catalog coverage (top-5, 5 queries): 0.63%


## 11. Saving — Sparse Similarity + Model Artifacts

The full dense similarity matrix is **~180 MB** for 3k movies and grows quadratically.
We store only the **top-50 neighbours** per movie as a sparse matrix (~3 MB), then
save the dense version separately for notebooks that need it.

In [48]:
n = len(new_df)
K_NEIGHBOURS = 50

sparse_sim = lil_matrix((n, n), dtype=np.float32)
for i in range(n):
    top_k_idx = np.argsort(similarity[i])[::-1][1 : K_NEIGHBOURS + 1]
    sparse_sim[i, top_k_idx] = similarity[i, top_k_idx]

sparse_csr = sparse_sim.tocsr()
sp.save_npz('similarity_sparse.npz', sparse_csr)
print(f"Sparse matrix saved  ({K_NEIGHBOURS} neighbours/movie)")

# Also pickle full matrix + dataframe for notebook use
pickle.dump(new_df,     open('movie_list.pkl', 'wb'))
pickle.dump(similarity, open('similarity.pkl',  'wb'))
print("Dense pickle saved   (movie_list.pkl, similarity.pkl)")

print("\nAll artifacts saved:")
print("  movie_list.pkl       — processed movie dataframe")
print("  similarity.pkl       — full cosine similarity matrix")
print("  similarity_sparse.npz — top-50 sparse matrix (small & fast)")

Sparse matrix saved  (50 neighbours/movie)
Dense pickle saved   (movie_list.pkl, similarity.pkl)

All artifacts saved:
  movie_list.pkl       — processed movie dataframe
  similarity.pkl       — full cosine similarity matrix
  similarity_sparse.npz — top-50 sparse matrix (small & fast)
